In [0]:
%>%sql

DESCRIBE bakery_pos_develop.silver.pos_session

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_pos_session AS
SELECT
    session_id,
    session_name,
    config_id,
    config_name,
    start_at,
    stop_at,
    cash_register_balance_end_real,
    cash_register_balance_start,
    created_at,
    updated_at,
    is_updated,
    is_deleted
FROM bakery_pos_develop.silver.pos_session
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY session_id
    ORDER BY updated_at DESC
) = 1

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_pos_session(
    session_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    session_id BIGINT,
    config_id INT,
    config_name STRING,
    start_at TIMESTAMP,
    stop_at TIMESTAMP,
    cash_register_balance_end_real DOUBLE,
    cash_register_balance_start DOUBLE,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY(session_sk)
) USING DELTA

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.dim_pos_session AS tgt
USING stg_pos_session AS src
ON tgt.session_id = src.session_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.config_id = src.config_id,
        tgt.config_name = src.config_name,
        tgt.start_at = src.start_at,
        tgt.stop_at = src.stop_at,
        tgt.cash_register_balance_end_real = src.cash_register_balance_end_real,
        tgt.cash_register_balance_start = src.cash_register_balance_start,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        session_id,
        config_id,
        config_name,
        start_at,
        stop_at,
        cash_register_balance_end_real,
        cash_register_balance_start,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.session_id,
        src.config_id,
        src.config_name,
        src.start_at,
        src.stop_at,
        src.cash_register_balance_end_real,
        src.cash_register_balance_start,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )